In [1]:
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.network import (
    LigandReceptorNetwork,
    WeightedNetwork
)
from nichenetpy.ann_utils import prepare_ann
from nichenetpy.gene_symbol import mouse_alias_info

import anndata
import numpy as np
import os
import requests
import pickle

# Basic NicheNetPy tutorial

Here we provide some basic information on how to use NichenetPy, for more in-depth information please see the more advanced tutorials described in the [README](../README.md)

## NicheNet Model

You can download the NicheNet model from Zenodo

In [2]:
filename = "nichenet_mouse.pkl"
file_path = os.path.join(
    "./tutorial_files", # set a directory to store the model here
    filename
)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/17061000/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Read the file

In [3]:
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())

The model is stored in a pickle file, this is a native format that allows you to store python objects. This particular pickle file contains a dictionary (key-value mapping). You can check the dictionary keys as follows...

In [4]:
list(model.keys())

['predictor', 'lr_network', 'lr_sig', 'gr']

### access the model components

As you can see the dictionary contains the ligand-receptor network, the weighted networks (see [model construction tutorial](model_construction.ipynb)) and the ligand activity predictor. 

In [5]:
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor : LigandActivityPredictor = model["predictor"]
lr_network : LigandReceptorNetwork = model["lr_network"]
lr_sig : WeightedNetwork = model["lr_sig"]
gr : WeightedNetwork = model["gr"]

### Ligand Activity Predictor

The ligand activity predictor is a wrapper around the ligand-target matrix. You can access the ligand-target matrix as follows. 

In [7]:
lt_matrix = predictor.ligand_target_matrix
row_names = predictor.get_genes()
col_names = predictor.get_ligands()

For quick help you can use python's builtin help function. It is however more convenient to look up the documentation online. 

In [18]:
help(predictor)

Help on LigandActivityPredictor in module nichenetpy.prediction object:

class LigandActivityPredictor(builtins.object)
 |  LigandActivityPredictor(ligand_target_matrix: scipy.sparse._csr.csr_matrix | scipy.sparse._csc.csc_matrix | numpy.ndarray, row_names: list[str] | tuple[str], col_names: list[str] | tuple[str]) -> None
 |
 |  This class facilitates the computation of ligand activities using a ligand-target matrix.
 |
 |  Parameters
 |  ----------
 |  ligand_target_matrix : numpy.ndarray
 |      a (ngenes X nligands) matrix describing the potential that a ligand may regulate a target gene
 |  row_names : list of str
 |      list of names of the rows/genes
 |  col_names : list of str
 |      list of names of the columns/ligands
 |
 |  Raises
 |  ------
 |  TypeError
 |      if the arguments have the wrong type
 |
 |  Attributes
 |  ----------
 |  ligand_target_matrix : numpy.ndarray
 |      a (ngenes X nligands) matrix describing the potential that a ligand may regulate a target gene

## AnnData Objects

Now let's take a look at an AnnData object

In [8]:
data_path = os.path.normpath("./tutorial_files/AnnData") # set a directory to store AnnData object here
if not os.path.exists(data_path):
    os.makedirs(data_path)
filename = "annData3531889.h5"
file_path = os.path.join(data_path, filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/15574665/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

In [9]:
ann = anndata.io.read_h5ad(os.path.join(data_path, "annData3531889.h5"))

The observations contain information about the samples. To run a NicheNet analysis the samples should be cells and they should be annotated with at least the celltype and the condition. In this case we have the columns "celltype" and "aggregate". 

In [10]:
ann.obs.head()

,nGene,nUMI,orig.ident,aggregate,res.0.6,celltype,nCount_RNA,nFeature_RNA
W380370,880.0,1611.0,LN_SS,SS,1,CD8 T,1607.0,876
W380372,541.0,891.0,LN_SS,SS,0,CD4 T,885.0,536
W380374,742.0,1229.0,LN_SS,SS,0,CD4 T,1223.0,737
W380378,847.0,1546.0,LN_SS,SS,1,CD8 T,1537.0,838
W380379,839.0,1606.0,LN_SS,SS,0,CD4 T,1603.0,836


This AnnData object contains a "counts", "data" and "scale.data" layer. 

In [11]:
ann.layers.keys()

KeysView(Layers with keys: counts, data, scale.data)

The X attribute is often used to store the default data, however NicheNetPy generally expects your data to be stored in a layer. 

In [12]:
ann.X

NicheNetPy expects to find the gene names to be stored in var_names. 

In [13]:
ann.var_names

Index(['0610005C13Rik', '0610007C21Rik', '0610007L01Rik', '0610007P08Rik',
       '0610007P14Rik', '0610007P22Rik', '0610009B22Rik', '0610009D07Rik',
       '0610009L18Rik', '0610009O20Rik',
       ...
       'Zpbp2', 'Zscan18', 'Zscan2', 'Zwilch', 'mmu-mir-2134-1',
       'mmu-mir-2134-2', 'snoU109', 'snoU97', 'snoZ39', 'snoZ40'],
      dtype='object', length=13541)

In case you have trouble running a NicheNet analysis on your AnnData object you can try prepare_ann. 

In [14]:
prepare_ann(ann)

You can check which percentage of your genes are present in the ligand-target matrix. 

In [15]:
predictor.gene_presence(ann.var_names)

0.7487630160254043

It helps to convert gene symbol aliases. 

In [16]:
mouse_alias_info.alias_to_symbol(ann)

In [17]:
predictor.gene_presence(ann.var_names)

0.8348718706151688

The simplest way to run a NicheNet analysis is to [use the wrapper function](wrapper.ipynb). We also provide a [step-by-step tutorial](steps.ipynb) and several usecases. All notebooks are documented in the [README](../README.md). 